# 05a — Facet similarity

Compare DayHikes on Facets. Each Facet gives a FacetScore in [0, 1] for a pair
of DayHikes: 1 alike, 0 unlike. The Hiker's FacetWeights combine them into one
score with a coverage figure. This is the notebook design phase of ADR-0011
(proposed); production Ranking stays ADR-0003 cosine.

Facts come from `{catalog}.similarity_facts.pkl`, written by
`05_similarity_facts.ipynb`. When that table is absent (the committed OSM
fixture `tests/fixtures/catalog.osm`, and always under pytest) this notebook
builds the facts itself with the same `dayhike.similarity` helpers.

**Facets here:** Effort, Altitude, Profile, Landscape, Weather, Water,
Facilities, Trail. After them: one Seed ranked under two FacetWeights sets,
several Seeds (the maximum rule), how often each Facet is missing, the
verification summary, and the design points for review.

Each Facet section: the measures and their constants with the region's spread,
then verification. A verification states the expected verdict and the reason
before the score is computed, then prints a grep-able line
`verify<TAB><facet><TAB><check><TAB><score><TAB>PASS|FAIL`.

In [ ]:
import os
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from dayhike import build_catalog, geometry_store_path_for, sample_climate
from dayhike.similarity import (
    FACILITIES_FLAG_COLUMNS,
    LANDCOVER_SHARE_COLUMNS,
    SURFACE_SHARE_COLUMNS,
    altitude_facts,
    combined_score,
    effort_facts,
    facet_score,
    facilities_facts,
    landscape_facts,
    load_region,
    mix,
    multi_seed_score,
    presence,
    profile_dtw,
    profile_facts,
    profile_lockstep,
    ratio,
    similarity_facts_path_for,
    tolerance,
    trail_facts,
    water_facts,
    weather_facts,
)

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
on_fixture = not (
    local.is_file()
    and geometry_store_path_for(local).is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
)
if on_fixture:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(fixture, catalog_path)
    # The fixture has no ClimateNormal. As in 03, a tiny synthetic WorldClim
    # (one value per month over the fixture box) is sampled so the Weather
    # checks have values to compare. There is no DEM tile: southness stays missing.
    import rasterio
    from rasterio.transform import from_bounds

    climate_dir = catalog_path.parent / "climate"
    climate_dir.mkdir()
    for month in range(1, 13):
        for variable, value in (("tavg", 2.0 * month), ("prec", 60.0 + month), ("srad", 1_500.0 * month)):
            with rasterio.open(
                climate_dir / f"wc2.1_30s_{variable}_{month:02d}.tif",
                "w",
                driver="GTiff",
                height=8,
                width=8,
                count=1,
                dtype="float32",
                crs="EPSG:4326",
                transform=from_bounds(14.0, 44.0, 16.0, 47.0, 8, 8),
                nodata=-9999.0,
            ) as dataset:
                dataset.write(np.full((8, 8), value, dtype="float32"), 1)
    sample_climate(catalog_path, climate_dir)
else:
    catalog_path = local
dem_dir = catalog_path.parent / "dem"  # FABDEM tiles; none next to the fixture
print("catalog", catalog_path)

## Load the facts

Read the facts table when it exists. Otherwise load the region (lat 42.3–46.9,
lon 13.3–19.5) and build the same table as `05`.

The reversed-direction checks need vertex chains, which the table does not
hold. With the table present, `vertices` and `dayhikes` are a small Julian Alps
box (lat 46.3–46.5, lon 13.6–14.0) used only for those checks; without it they
are the whole region.

In [ ]:
REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
CHECK_LAT = (46.3, 46.5)
CHECK_LON = (13.6, 14.0)

facts_path = similarity_facts_path_for(catalog_path)
if not on_fixture and facts_path.is_file():
    facts = pd.read_pickle(facts_path)
    vertices, amenities, dayhikes, ways = load_region(
        catalog_path, lat_range=CHECK_LAT, lon_range=CHECK_LON
    )
    print("facts read from", facts_path)
else:
    vertices, amenities, dayhikes, ways = load_region(
        catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
    )
    facts = gpd.GeoDataFrame(
        dayhikes.set_index("id")[
            ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
        ]
        .join(effort_facts(dayhikes, vertices))
        .join(altitude_facts(dayhikes))
        .join(profile_facts(dayhikes, vertices))
        .join(landscape_facts(dayhikes, vertices, amenities))
        .join(weather_facts(dayhikes, vertices, dem_dir))
        .join(water_facts(dayhikes, vertices, amenities))
        .join(facilities_facts(dayhikes, vertices, amenities))
        .join(trail_facts(dayhikes, vertices, ways)),
        geometry="geometry",
        crs=dayhikes.crs,
    )
    print("facts built from the region")
print(f"facts_rows\t{len(facts)}")
print(f"check_dayhikes\t{len(dayhikes)}")

## Measure kinds and combining

All functions are in `dayhike.similarity`. Each returns a value in [0, 1], or
None when either DayHike lacks the fact.

| Kind | Rule | Used for |
| --- | --- | --- |
| `ratio` | min(a′, b′) / max(a′, b′), a′ = max(a, floor) | sizes: 15 vs 20 km is closer than 5 vs 10 km |
| `tolerance` | max(0, 1 − abs(a − b) / D) | levels: 1800 vs 2000 m is close, 300 vs 2000 m is not |
| `mix` | Σ min(pᵢ, qᵢ) over fractions summing to 1 | LandCover (Landscape), SurfaceMix (Trail) |
| `presence` | share of yes/no flags that agree | Peak summited, Viewpoint along (Landscape), the five Facilities flags |

- FacetScore (`facet_score`): the plain mean of the Facet's measures that are
  not missing. A missing Difficulty drops out; length and gain still count.
- Combined score (`combined_score`): Σ wᶠ·sᶠ / Σ wᶠ over Facets with a
  FacetScore and wᶠ > 0. FacetWeights can be in any scale (50/30/20 or 5/3/2);
  weight 0 switches a Facet off. Coverage is the scored weight over all positive
  weight. No scored Facet gives a missing score.
- Several Seeds (`multi_seed_score`): a candidate's best combined score to any
  one Seed, never a mix of Seeds.

`measure_spread` prints a measure's spread in the region and, for a fixed
sample of random pairs, the median score its constant gives.

In [ ]:
rng = np.random.default_rng(0)
left_ids = rng.choice(facts.index.to_numpy(), 2000)
right_ids = rng.choice(facts.index.to_numpy(), 2000)


def measure_spread(column, measure, constant):
    values = facts[column]
    q = values.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(1).tolist()
    pair_scores = [
        measure(facts.at[a, column], facts.at[b, column])
        for a, b in zip(left_ids, right_ids)
    ]
    pair_scores = [s for s in pair_scores if s is not None]
    median = float(np.median(pair_scores)) if pair_scores else float("nan")
    print(
        f"measure\t{column}\t{constant}\tmissing={values.isna().mean():.3f}\t"
        f"q05,q25,q50,q75,q95={q}\trandom_pair_median={median:.2f}"
    )

### Pair verification helpers

- `pair_rows` looks up the named DayHikes (each must be a unique `osm_name`) in
  the order similar A, similar B, dissimilar A, dissimilar B.
- `show_pairs` prints their facts and draws each DayHike line on a small
  topographic map (the line alone when offline).
- `verify_pairs` scores both pairs. Both `verify` lines PASS when the similar
  pair scores higher than the dissimilar one. The soft targets (similar ≥ 0.7,
  dissimilar ≤ 0.4) are printed on separate `soft` lines. Both kinds are kept
  for the verification summary at the end.

In [ ]:
def pair_rows(similar_names, dissimilar_names):
    names = [*similar_names, *dissimilar_names]
    named = facts[(facts["name_origin"] == "osm_name") & facts["name"].isin(names)]
    assert sorted(named["name"]) == sorted(set(names)), "pair names must be unique osm_name DayHikes"
    return named.set_index("name").loc[names]


def show_pairs(pairs, columns, title):
    print(pairs[columns].round(1).to_string())
    fig, axes = plt.subplots(1, len(pairs), figsize=(3 * len(pairs), 3))
    for ax, (name, row) in zip(axes, pairs.iterrows()):
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
        one.plot(ax=ax, color="#c2410c", linewidth=2)
        try:
            import contextily as cx

            cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
        except Exception as error:  # offline: the line alone is still the map
            print("basemap skipped", type(error).__name__)
        ax.set_title(name, fontsize=8)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    show_fig(fig)


VERIFY = {}  # (facet, check) -> (score, PASS or FAIL), read by the summary at the end
SOFT = {}  # (facet, check) -> (score, target, met)


def verify(facet, check, score, ok):
    result = "PASS" if ok else "FAIL"
    VERIFY[(facet, check)] = (score, result)
    print(f"verify\t{facet}\t{check}\t{score:.3f}\t{result}")


def verify_pairs(facet, score_fn, pairs):
    similar = score_fn(pairs.iloc[0], pairs.iloc[1])
    dissimilar = score_fn(pairs.iloc[2], pairs.iloc[3])
    ok = similar > dissimilar
    verify(facet, "similar_pair", similar, ok)
    verify(facet, "dissimilar_pair", dissimilar, ok)
    for check, score, target, met in (
        ("similar_pair", similar, ">=0.7", similar >= 0.7),
        ("dissimilar_pair", dissimilar, "<=0.4", dissimilar <= 0.4),
    ):
        SOFT[(facet, check)] = (score, target, met)
        print(f"soft\t{facet}\t{check}\t{score:.3f}\t{target}\t{'met' if met else 'missed'}")

### Synthetic check helpers

`synthetic_checks` runs three checks for one Facet and prints one `verify`
line each:

- identity: the lowest score of a DayHike against itself, which must be 1.
- symmetry: the largest gap between A vs B and B vs A over the random pairs,
  which must be 0.
- reversed: the lowest score of a DayHike's facts against the same DayHike's
  facts rebuilt from its reversed vertex chain (`reversed_vertices`), which
  must be 1.

In [ ]:
def reversed_vertices(vertices):
    last = vertices.groupby("dayhike_id")["seq"].transform("max")
    return vertices.assign(seq=last - vertices["seq"])


def synthetic_checks(facet, score_fn, forward, backward):
    """Identity, symmetry and reversed direction for one Facet."""
    identity = [score_fn(row, row) for _, row in facts.iterrows()]
    identity = [s for s in identity if s is not None]
    worst = min(identity)
    verify(facet, "identity", worst, abs(worst - 1.0) < 1e-9)

    gaps = []
    for a, b in zip(left_ids, right_ids):
        ab, ba = score_fn(facts.loc[a], facts.loc[b]), score_fn(facts.loc[b], facts.loc[a])
        if ab is not None:
            gaps.append(abs(ab - ba))
    gap = max(gaps)
    verify(facet, "symmetry_max_gap", gap, gap < 1e-9)

    turned = [score_fn(forward.loc[i], backward.loc[i]) for i in forward.index]
    turned = [s for s in turned if s is not None]
    worst = min(turned)
    verify(facet, "reversed", worst, abs(worst - 1.0) < 1e-9)


backward_vertices = reversed_vertices(vertices)

## Effort

Three measures: length, elevation gain, Difficulty rank.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `length_m` | ratio | floor 1 km | The Catalog length band starts at 1 km, so the floor only matters for a hand-made input. 6 km vs 18 km scores 0.33. |
| `elevation_gain_m` | ratio | floor 100 m | About 9% of region DayHikes climb under 100 m. Those are flat walks and should match each other (10 m vs 30 m scores 1), not sit at the far end of a ratio. |
| `difficulty_rank` | tolerance | D = 3 | Tagged on about half of the region, mostly SAC 1–2. One grade apart scores 0.67; T2 vs T5 scores 0. |

The data does not argue for other values, so the PRD starting constants stay.

In [ ]:
LENGTH_FLOOR_M = 1_000
GAIN_FLOOR_M = 100
DIFFICULTY_D = 3


def effort_score(a, b):
    return facet_score(
        [
            ratio(a["length_m"], b["length_m"], floor=LENGTH_FLOOR_M),
            ratio(a["elevation_gain_m"], b["elevation_gain_m"], floor=GAIN_FLOOR_M),
            tolerance(a["difficulty_rank"], b["difficulty_rank"], d=DIFFICULTY_D),
        ]
    )


measure_spread("length_m", lambda a, b: ratio(a, b, floor=LENGTH_FLOOR_M), f"ratio floor={LENGTH_FLOOR_M}")
measure_spread("elevation_gain_m", lambda a, b: ratio(a, b, floor=GAIN_FLOOR_M), f"ratio floor={GAIN_FLOOR_M}")
measure_spread("difficulty_rank", lambda a, b: tolerance(a, b, d=DIFFICULTY_D), f"tolerance D={DIFFICULTY_D}")

### Effort verification (real region)

Expected before computing:

- **Similar:** "Zagorje - Završje - Kum" (Slovenia) and "Mosor-Kosa" (Dalmatia).
  Both are 6–7 km with 1,000–1,250 m of climbing at SAC 2: a solid half-day
  up a hill. They are far apart and look nothing alike, which Effort ignores.
- **Dissimilar:** "Pot kralja Triglava" and "Log - Mangart". The first is a
  3 km, under-300 m T1 walk; the second an 11 km, 2,800 m climb at T6.

Skipped on the fixture, which has no such DayHikes.

In [ ]:
if on_fixture:
    print("effort pairs skipped on the fixture")
else:
    effort_pairs = pair_rows(
        ["Zagorje - Završje - Kum", "Mosor-Kosa"],
        ["Pot kralja Triglava", "Log - Mangart"],
    )
    show_pairs(effort_pairs, ["length_m", "elevation_gain_m", "difficulty_rank"], "Effort pairs")
    verify_pairs("effort", effort_score, effort_pairs)

### Effort synthetic checks

- identity: every DayHike against itself scores 1 (worst case printed).
- symmetry: A vs B equals B vs A on 2,000 random pairs (largest gap printed).
- reversed: Effort facts rebuilt from the reversed vertex chain score 1 against
  the stored direction (worst case printed). This is the check that keeps
  elevation gain direction-free.

In [ ]:
synthetic_checks(
    "effort",
    effort_score,
    effort_facts(dayhikes, vertices),
    effort_facts(dayhikes, backward_vertices),
)

## Altitude

Two measures: highest and lowest point.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `highest_m` | tolerance | D = 1000 m | Region tops run from the coast to about 2,800 m; the middle half spans about 440–1,440 m. 1,800 vs 2,000 m scores 0.8; a lowland hill vs a 2,500 m ridge scores 0. |
| `lowest_m` | tolerance | D = 1000 m | Lowest points run from sea level to about 2,400 m; the middle half spans about 250–900 m. A climb from the sea vs a walk from a 1,500 m pass scores 0. |

A random pair of region DayHikes differs by about 600 m on the highest point
and about 400 m on the lowest, so D = 1000 m puts a typical random pair near
0.4 and 0.6 and keeps pairs from the same kind of terrain high. The PRD
starting constants stay.

In [ ]:
HIGHEST_D_M = 1_000
LOWEST_D_M = 1_000


def altitude_score(a, b):
    return facet_score(
        [
            tolerance(a["highest_m"], b["highest_m"], d=HIGHEST_D_M),
            tolerance(a["lowest_m"], b["lowest_m"], d=LOWEST_D_M),
        ]
    )


measure_spread("highest_m", lambda a, b: tolerance(a, b, d=HIGHEST_D_M), f"tolerance D={HIGHEST_D_M}")
measure_spread("lowest_m", lambda a, b: tolerance(a, b, d=LOWEST_D_M), f"tolerance D={LOWEST_D_M}")

### Altitude verification (real region)

Expected before computing:

- **Similar:** "Vršič - Razor" and "Kotovo sedlo - Mangart". Both start from a
  high Julian Alps road (about 1,600–1,700 m) and top out at about 2,500 m.
- **Dissimilar:** "Omiš - Kula" and "Dolič - Triglav". Omiš starts at the
  Dalmatian coast and tops out below 900 m; the Triglav summit route starts
  above 2,100 m and reaches the highest point in the region.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("altitude pairs skipped on the fixture")
else:
    altitude_pairs = pair_rows(
        ["Vršič - Razor", "Kotovo sedlo - Mangart"],
        ["Omiš - Kula", "Dolič - Triglav"],
    )
    show_pairs(altitude_pairs, ["highest_m", "lowest_m"], "Altitude pairs")
    verify_pairs("altitude", altitude_score, altitude_pairs)

### Altitude synthetic checks

Same three checks. Highest and lowest point do not depend on the walking
direction, so the reversed check confirms the facts are read the same way.

In [ ]:
synthetic_checks(
    "altitude",
    altitude_score,
    altitude_facts(dayhikes),
    altitude_facts(dayhikes),
)

## Profile

One measure: the ProfileShape (`profile_shape`, built as in `05`: 64 points
along the distance fraction, height over max(range, 50 m)). Two ways to compare
two ProfileShapes, both taking the better of the candidate walked forward or
reversed:

| Method | Rule | Constant |
| --- | --- | --- |
| lock-step (`profile_lockstep`) | 1 − mean abs difference, point i against point i | 64 points, range floor 50 m |
| banded DTW (`profile_dtw`) | 1 − mean abs difference along the cheapest warping path | Sakoe–Chiba band 10% of the points (6) |

DTW lets a climb at 30% of the length line up with a climb at 50%, as far as
the band allows. It never scores below lock-step on the same pair, because the
diagonal is one of its paths. The question is whether that slack separates
similar from dissimilar pairs better, or only lifts every score.

Both scores stay in [0, 1] because every height does. Below, the spread of
each method over the same random pairs as the other Facets.

In [ ]:
profile_methods = {"lockstep": profile_lockstep, "dtw": profile_dtw}
for method, fn in profile_methods.items():
    pair_scores = [
        fn(facts.at[a, "profile_shape"], facts.at[b, "profile_shape"])
        for a, b in zip(left_ids, right_ids)
    ]
    pair_scores = [s for s in pair_scores if s is not None]
    q = np.quantile(pair_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
    print(
        f"measure\tprofile_shape\t{method}\tmissing={facts['profile_shape'].isna().mean():.3f}\t"
        f"random_pair_q05,q25,q50,q75,q95={q}"
    )

### Profile verification (real region)

Expected before computing. Three pairs that should be similar:

- "Koča pri Savici - Dom na Komni" and "Zavala - Sv. Ante": each is one
  straight climb from bottom to top, but 1,400 m over 6.4 km in the Julian
  Alps against 530 m over 2.5 km up from the sea on Hvar. Zavala is stored top
  to bottom, so it only matches walked reversed.
- "Gozdna učna pot Planina–Mirna gora" and "Maslinovi puti": both go up to one
  top and back down. The first tops out about two thirds of the way along, the
  second halfway. This is the pair DTW is meant for.
- "Šturmovci - Ptujsko jezero" and "P16": both flat (under 10 m of climbing).
  A few metres of noise must not make them look different.

Three that should not:

- "Koča pri Savici - Dom na Komni" and "Dom Paklenica - Jurline": one climb
  against a route that climbs and drops several times.
- "Maslinovi puti" and "Blejski razgledni trail": one top against the Bled
  viewpoint trail, which climbs to several viewpoints in turn.
- "Šturmovci - Ptujsko jezero" and "Zavala - Sv. Ante": flat against a
  straight climb.

The table scores all six pairs with both methods. A method separates better
when its margin (lowest similar score minus highest dissimilar score) is
larger; the mean gap between the two groups is printed too. The `verify` lines
below use the kept method on the first pair of each group: the same Komna climb
against another single climb and against a rolling route.

Skipped on the fixture.

In [ ]:
PROFILE_PAIRS = [
    ("similar", "Koča pri Savici - Dom na Komni", "Zavala - Sv. Ante"),
    ("similar", "Gozdna učna pot Planina–Mirna gora", "Maslinovi puti"),
    ("similar", "Šturmovci - Ptujsko jezero", "P16"),
    ("dissimilar", "Koča pri Savici - Dom na Komni", "Dom Paklenica - Jurline"),
    ("dissimilar", "Maslinovi puti", "Blejski razgledni trail"),
    ("dissimilar", "Šturmovci - Ptujsko jezero", "Zavala - Sv. Ante"),
]

if on_fixture:
    print("profile pairs skipped on the fixture")
else:
    profile_rows = pair_rows(sorted({name for _, a, b in PROFILE_PAIRS for name in (a, b)}), [])
    profile_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            **{
                method: fn(profile_rows.at[a, "profile_shape"], profile_rows.at[b, "profile_shape"])
                for method, fn in profile_methods.items()
            },
        }
        for expected, a, b in PROFILE_PAIRS
    )
    print(profile_table.round(3).to_string())
    for method in profile_methods:
        by_group = profile_table.groupby("expected")[method]
        margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
        gap = by_group.mean()["similar"] - by_group.mean()["dissimilar"]
        print(f"profile_separation\t{method}\tmargin={margin:.3f}\tmean_gap={gap:.3f}")

    fraction = np.linspace(0.0, 1.0, 64)
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharey=True)
    for ax, (expected, a, b) in zip(axes.flat, PROFILE_PAIRS):
        for name, color in ((a, "#0f766e"), (b, "#c2410c")):
            ax.plot(fraction, profile_rows.at[name, "profile_shape"], color=color, label=name)
        ax.set_title(expected, fontsize=9)
        ax.set_xlabel("fraction of length")
        ax.legend(fontsize=7)
    fig.suptitle("Profile pairs: ProfileShape in stored direction")
    fig.tight_layout()
    show_fig(fig)

### Profile method kept: lock-step

The design keeps lock-step. The rule was: DTW only if it separates the
verification pairs better. On the real region it does not (printed above):

| Method | Lowest similar | Highest dissimilar | Margin | Mean gap | Random-pair median |
| --- | --- | --- | --- | --- | --- |
| lock-step | 0.870 | 0.721 | 0.148 | 0.303 | 0.77 |
| banded DTW | 0.975 | 0.854 | 0.121 | 0.258 | 0.85 |

DTW does what it is for on the shifted-top pair (Planina–Mirna gora against
Maslinovi puti: 0.870 → 0.975). It lifts the dissimilar pairs more on average
(+0.08 against +0.04 for the similar ones): the single top against the Bled
viewpoint trail goes 0.721 → 0.854, because
the band lets one climb of the rolling route line up with the single climb.
Every random pair moves up too, so DTW squeezes the useful range of the Facet
toward 1. The Profile FacetScore below (`profile_score`) is lock-step;
`profile_dtw` stays in `dayhike.similarity` for this comparison.

Profile scores sit high in general (a random pair's median is 0.77): two
ProfileShapes that both go up and down share much of the unit square. The
soft target dissimilar ≤ 0.4 is therefore out of reach: even flat against a
straight climb scores 0.52. The verdict rests on similar > dissimilar.

In [ ]:
def profile_score(a, b):
    return facet_score([profile_lockstep(a["profile_shape"], b["profile_shape"])])


if on_fixture:
    print("profile verify pairs skipped on the fixture")
else:
    profile_pairs = pair_rows(
        ["Koča pri Savici - Dom na Komni", "Zavala - Sv. Ante"],
        ["Koča pri Savici - Dom na Komni", "Dom Paklenica - Jurline"],
    )
    show_pairs(profile_pairs, ["length_m", "elevation_gain_m", "highest_m", "lowest_m"], "Profile pairs")
    verify_pairs("profile", profile_score, profile_pairs)

### Profile synthetic checks

Same three checks. For reversed, the ProfileShape is rebuilt from the reversed
vertex chain, which mirrors it; taking the better of forward and reversed must
give 1.

In [ ]:
synthetic_checks(
    "profile",
    profile_score,
    profile_facts(dayhikes, vertices),
    profile_facts(dayhikes, backward_vertices),
)

## Landscape

Four measures: the LandCover mix, Peak summited, Viewpoint along, Peaks around.
The facts are built as in `05` (six LandCover shares including the leftover;
Peak within 50 m; Viewpoint in the 100 m Corridor; Peaks within 2 km, all
measured to the line).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| LandCover mix (six shares) | mix | none | Scores the shared share: half forest and half grass against all forest is 0.5. The leftover share keeps hikes through cropland or towns from looking like forest hikes with a few missing percent (it averages 7.5%, 95th percentile 34%). A random pair of region DayHikes shares a median 0.69, because forest is 65% of the average hike; forest against bare rock is near 0. |
| `peak_summited` | presence | none | About three in ten region DayHikes pass a Peak. Both-absent counts as agreement, so two hikes with no summit agree on this point. |
| `viewpoint_along` | presence | none | About a quarter of region DayHikes have a Viewpoint in the Corridor; same rule. |
| `peaks_around` | ratio | floor 1 | Counts run from 0 to about 90 (middle half 4–18). Ratio, not a difference, so 20 vs 40 Peaks is as far apart as 2 vs 4. The floor of 1 treats "none" as "one" against a ratio, so 0 vs 3 scores 0.33, not 0. A random pair's median is 0.33. |

The Landscape FacetScore (`landscape_score`) is the plain mean of the four. The
PRD starting constant (floor 1) stays; the data does not argue for another.
Presence measures lift every score: two hikes that both have no summit and no
Viewpoint share 2 of the 4 points before the LandCover mix and Peak count are
compared, which is why the dissimilar pairs below do not reach 0.

Result on the real region (below): the main similar pair scores 0.94 and the
dissimilar one 0.26. Forest against open karst ("Zahum Idovac") scores 0.58:
the mixes share 0.04, but neither hike has a summit or Viewpoint, and they
agree on that. The soft target dissimilar ≤ 0.4 is met for forest against
alpine, not for forest against karst, which is the presence effect above, not
a LandCover error. The verdict rests on similar > dissimilar.

In [ ]:
PEAKS_AROUND_FLOOR = 1
LANDCOVER = list(LANDCOVER_SHARE_COLUMNS)


def landscape_score(a, b):
    return facet_score(
        [
            mix([a[c] for c in LANDCOVER], [b[c] for c in LANDCOVER]),
            presence([a["peak_summited"]], [b["peak_summited"]]),
            presence([a["viewpoint_along"]], [b["viewpoint_along"]]),
            ratio(a["peaks_around"], b["peaks_around"], floor=PEAKS_AROUND_FLOOR),
        ]
    )


mix_scores = [
    mix([facts.at[a, c] for c in LANDCOVER], [facts.at[b, c] for c in LANDCOVER])
    for a, b in zip(left_ids, right_ids)
]
mix_scores = [s for s in mix_scores if s is not None]
q = np.quantile(mix_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist() if mix_scores else []
print(
    f"measure\tlandcover_mix\tmix\tmissing={facts[LANDCOVER].isna().any(axis=1).mean():.3f}\t"
    f"random_pair_q05,q25,q50,q75,q95={q}"
)
for column in LANDCOVER:
    print(f"share\t{column}\tmean={facts[column].mean():.3f}\tq95={facts[column].quantile(0.95):.3f}")
for flag in ("peak_summited", "viewpoint_along"):
    print(f"measure\t{flag}\tpresence\ttrue_share={facts[flag].mean():.3f}")
measure_spread("peaks_around", lambda a, b: ratio(a, b, floor=PEAKS_AROUND_FLOOR), f"ratio floor={PEAKS_AROUND_FLOOR}")

### Landscape verification (real region)

Expected before computing:

- **Similar:** "Begunje - Cemarovnik - Velika Špička" and "Rakitna - Vrbica".
  Both are forested (86–93% tree) hikes to a Peak with a Viewpoint on the way,
  among a couple of dozen Peaks. They are far apart, which Landscape ignores.
- **Dissimilar:** "Planinarska staza 16" and "Gorjanska pot". The first is a
  98% forest path with no Peak and no Viewpoint nearby; the second is 99%
  bare rock to a summit among more than 40 Peaks.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("landscape pairs skipped on the fixture")
else:
    landscape_pairs = pair_rows(
        ["Begunje - Cemarovnik - Velika Špička", "Rakitna - Vrbica"],
        ["Planinarska staza 16", "Gorjanska pot"],
    )
    show_pairs(
        landscape_pairs,
        [*LANDCOVER, "peak_summited", "viewpoint_along", "peaks_around"],
        "Landscape pairs",
    )
    verify_pairs("landscape", landscape_score, landscape_pairs)

### More Landscape pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Predmeja - Iztokova koča" and "Mošćenice-Kremenjak/Brgud": two
  forest walks (95–97% tree), no summit, no Viewpoint, a dozen or two Peaks
  around. "Dolič - Triglav" and "Špremova pot": two bare-rock summit routes in
  the Julian Alps (94–100% bare, 20–50 Peaks around).
- **Dissimilar:** the forest walk "Predmeja - Iztokova koča" against the bare
  rock route "Dolič - Triglav" (forest vs alpine), and against "Zahum Idovac"
  (97% grass, open karst, few Peaks).

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one. Skipped on the fixture.

In [ ]:
LANDSCAPE_MORE = [
    ("similar", "Predmeja - Iztokova koča", "Mošćenice-Kremenjak/Brgud"),
    ("similar", "Dolič - Triglav", "Špremova pot"),
    ("dissimilar", "Predmeja - Iztokova koča", "Dolič - Triglav"),
    ("dissimilar", "Predmeja - Iztokova koča", "Zahum Idovac"),
]

if on_fixture:
    print("more landscape pairs skipped on the fixture")
else:
    more_rows = pair_rows(sorted({n for _, a, b in LANDSCAPE_MORE for n in (a, b)}), [])
    more_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "landcover_mix": mix([more_rows.at[a, c] for c in LANDCOVER], [more_rows.at[b, c] for c in LANDCOVER]),
            "score": landscape_score(more_rows.loc[a], more_rows.loc[b]),
        }
        for expected, a, b in LANDSCAPE_MORE
    )
    print(more_table.round(3).to_string())
    by_group = more_table.groupby("expected")["score"]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    verify("landscape", "more_pairs_margin", margin, margin > 0)

### Landscape synthetic checks

Same three checks. For reversed, the Landscape facts are rebuilt from the
reversed vertex chain; the Corridor and every radius are measured to the line,
so the walking direction must not matter.

In [ ]:
synthetic_checks(
    "landscape",
    landscape_score,
    landscape_facts(dayhikes, vertices, amenities),
    landscape_facts(dayhikes, backward_vertices, amenities),
)

## Weather

Four measures: temperature, precipitation and solar radiation for the slice
that `QUERY_MONTH` selects, and SolarAspect southness. The facts are built as
in `05` (the ClimateNormal regime from `feature_vectors`, the twelve months
from `climate_months`, southness from FABDEM: the downslope direction at about
32 vertices, 1 due south, 0 due north, 0.5 east or west, slopes under 5° left
out).

`QUERY_MONTH` is the one QueryMonth setting (ADR-0010). `None` compares the
June–August regime: June–August mean temperature, mean monthly precipitation
over the year, June–August solar radiation. A month 1–12 compares that month's
three values instead. Southness does not change with the month.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| temperature (°C) | tolerance | D = 10 °C | June–August temperature runs from about 10 °C on the high Julian Alps to 22 °C on the Dalmatian coast (5th–95th percentile). D = 10 °C is that spread: an alpine hut against a coastal walk scores near 0, two hikes in the same valley near 1. A random pair's median is about 0.7. |
| precipitation (mm) | ratio | floor 10 mm | A size, so a ratio: 60 vs 120 mm is as far apart as 100 vs 200 mm. Regime values run 52–169 mm (5th–95th percentile); the wettest Julian Alps get three times the rain of the Pannonian plain. No region month is under 19 mm, so the floor only guards a hand-made input. |
| solar radiation (kJ m⁻² day⁻¹) | tolerance | D = 5,000 | From the region's spread: the June–August regime runs 17,800–22,700 (5th–95th percentile), so D = 5,000 is that spread. A random pair's median is 0.70. The same D is kept for single months. Winter radiation differs less in absolute terms (January 3,900–5,900), so January solar scores sit higher; that is the climate, not a different scale. |
| southness | tolerance | D = 1 | Already in [0, 1]. Region middle half 0.36–0.72, median 0.54; a random pair's median is 0.78. North against south (0 vs 1) scores 0, east against south 0.5. Missing on about 1% of region DayHikes, all flat lowland walks with no point steeper than 5°; it drops out of the mean there. |

The Weather FacetScore (`weather_score`) is the plain mean of the four. PRD
starting constants stay; the solar-radiation D is set from the regime spread
as above. Southness is one measure in four, so two hikes with the same climate
on opposite faces of one mountain still score about 0.7: the climate is the
same, only the sun differs.

In [ ]:
QUERY_MONTH = None  # 1-12 compares that month's ClimateNormal; None the June-August regime
TEMPERATURE_D_C = 10
PRECIPITATION_FLOOR_MM = 10
SOLAR_RADIATION_D = 5_000  # kJ m-2 day-1, the region's 5th-95th percentile spread of the regime
SOUTHNESS_D = 1


def climate_slice(month):
    """Temperature, precipitation and solar radiation columns for a QueryMonth."""
    suffix = "" if month is None else f"_{month:02d}"
    return [f"{name}{suffix}" for name in ("temperature_c", "precipitation_mm", "solar_radiation")]


def weather_score(a, b, month=QUERY_MONTH):
    temperature, precipitation, solar = climate_slice(month)
    return facet_score(
        [
            tolerance(a[temperature], b[temperature], d=TEMPERATURE_D_C),
            ratio(a[precipitation], b[precipitation], floor=PRECIPITATION_FLOOR_MM),
            tolerance(a[solar], b[solar], d=SOLAR_RADIATION_D),
            tolerance(a["southness"], b["southness"], d=SOUTHNESS_D),
        ]
    )


temperature, precipitation, solar = climate_slice(QUERY_MONTH)
print(f"query_month\t{QUERY_MONTH}")
measure_spread(temperature, lambda a, b: tolerance(a, b, d=TEMPERATURE_D_C), f"tolerance D={TEMPERATURE_D_C}")
measure_spread(precipitation, lambda a, b: ratio(a, b, floor=PRECIPITATION_FLOOR_MM), f"ratio floor={PRECIPITATION_FLOOR_MM}")
measure_spread(solar, lambda a, b: tolerance(a, b, d=SOLAR_RADIATION_D), f"tolerance D={SOLAR_RADIATION_D}")
measure_spread("southness", lambda a, b: tolerance(a, b, d=SOUTHNESS_D), f"tolerance D={SOUTHNESS_D}")
for column in ("solar_radiation", "solar_radiation_01"):
    q05, q95 = facts[column].quantile([0.05, 0.95])
    print(f"spread\t{column}\tq05={q05:.0f}\tq95={q95:.0f}\tq95-q05={q95 - q05:.0f}")
print(f"monthly_precipitation_min\t{facts[[f'precipitation_mm_{m:02d}' for m in range(1, 13)]].min().min():.1f}")

### Weather verification (real region)

Expected before computing:

- **Similar:** "Kopiščarjeva pot" (the north face of Prisojnik, Julian Alps)
  and "Frischaufova pot" (Kamnik–Savinja Alps). Two shady, north-facing high
  alpine routes, both around 10 °C in summer, about 155 mm of rain a month and
  the region's lowest summer sun. They are 60 km apart in different ranges,
  which Weather ignores.
- **Dissimilar:** "Kopiščarjeva pot" and "Zavala - Sv. Ante" (Hvar). The
  island climb faces south from the sea, is 23 °C in summer with 62 mm of rain
  a month and the strongest sun in the region.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("weather pairs skipped on the fixture")
else:
    weather_pairs = pair_rows(
        ["Kopiščarjeva pot", "Frischaufova pot"],
        ["Kopiščarjeva pot", "Zavala - Sv. Ante"],
    )
    show_pairs(weather_pairs, [*climate_slice(QUERY_MONTH), "southness"], "Weather pairs")
    verify_pairs("weather", weather_score, weather_pairs)

### More Weather pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Zavala - Sv. Ante" and "Karst in Kornati": two sunny,
  south-facing Dalmatian coast walks (about 23 °C, 62 mm of rain a month).
  "Kopiščarjeva pot" and "Koča pri izviru Soče - Bavški Grintavec": two
  north-facing Julian Alps routes.
- **Dissimilar:** "Kopiščarjeva pot" against "Slovenska pot": the same
  mountain (Prisojnik), the same climate, opposite faces (north against
  south). Only southness differs, so it should score clearly below the
  similar pairs but not near 0 (about 0.7). And the north-facing "Koča v
  Grohotu - Raduha" (Kamnik–Savinja Alps) against "Planinarska staza
  Medvednica 3", a south-facing hill above Zagreb, 9 °C warmer with half the
  rain.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one. Skipped on the fixture.

In [ ]:
WEATHER_MORE = [
    ("similar", "Zavala - Sv. Ante", "Karst in Kornati"),
    ("similar", "Kopiščarjeva pot", "Koča pri izviru Soče - Bavški Grintavec"),
    ("dissimilar", "Kopiščarjeva pot", "Slovenska pot"),
    ("dissimilar", "Koča v Grohotu - Raduha", "Planinarska staza Medvednica 3"),
]

if on_fixture:
    print("more weather pairs skipped on the fixture")
else:
    more_rows = pair_rows(sorted({n for _, a, b in WEATHER_MORE for n in (a, b)}), [])
    more_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "southness": tolerance(more_rows.at[a, "southness"], more_rows.at[b, "southness"], d=SOUTHNESS_D),
            "score": weather_score(more_rows.loc[a], more_rows.loc[b]),
        }
        for expected, a, b in WEATHER_MORE
    )
    print(more_table.round(3).to_string())
    by_group = more_table.groupby("expected")["score"]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    verify("weather", "more_pairs_margin", margin, margin > 0)

### QueryMonth: the same pair in summer and in January

Expected before computing: "Zavala - Sv. Ante" (Hvar coast) and "K▲
(Hetvehely – Jakab-hegy – Cserkút)" (a south-facing Mecsek hill walk in
southern Hungary) are both warm (20–23 °C), dry and sunny in June–August, so
with no month they score fairly high (about 0.7). In January the coast stays
mild (about 8 °C) and the Pannonian hill is at freezing, so with
`QUERY_MONTH = 1` the score drops (about 0.55). Southness does not change with
the month.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("weather month pair skipped on the fixture")
else:
    month_pair = pair_rows(["Zavala - Sv. Ante", "K▲ (Hetvehely – Jakab-hegy – Cserkút)"], [])
    print(month_pair[[*climate_slice(None), *climate_slice(1), "southness"]].round(1).to_string())
    for month in (None, 1):
        score = weather_score(month_pair.iloc[0], month_pair.iloc[1], month=month)
        print(f"weather_month\t{month}\t{score:.3f}")

### Weather synthetic checks

Same three checks, on the slice `QUERY_MONTH` selects. For reversed, the
Weather facts are rebuilt from the reversed vertex chain. The climate is a
mean along the line, and southness reads the same vertices in either
direction (counted from the nearer end), so the walking direction must not
matter. On the fixture southness is missing (no DEM tile) and drops out; the
three climate measures are scored.

In [ ]:
synthetic_checks(
    "weather",
    weather_score,
    weather_facts(dayhikes, vertices, dem_dir),
    weather_facts(dayhikes, backward_vertices, dem_dir),
)

## Water

Two measures: the capped number of WaterSources in the Corridor and the
DryStretch. The facts are built as in `05` (drinking water, springs, taps and
water points within 100 m of the line; each placed along the walk; the DryStretch is the
longest stretch between them, from the first vertex to the last).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `water_count`, capped at 3 | tolerance | cap 3, D = 3 | 60% of region DayHikes pass no WaterSource, 22% pass one, 9% two and 8% three or more (maximum 19). Past three more water adds nothing for a day hike, so the count is capped before it is compared. With D = 3, none against one scores 0.67, none against three 0; four points and nineteen are both "plenty" and score 1. A random pair's median is 0.67: 43% of random pairs share a level (mostly both none) and score 1, and most of the rest are one level apart. |
| `dry_stretch_m` | ratio | floor 1 km | A size, so a ratio: 3 km of dry walking against 6 km is as far apart as 6 against 12. Region middle half 2.1–5.9 km, median 3.5 km, 95th percentile 11.7 km, and a random pair's median is 0.50. The floor of 1 km treats any stretch under it as the same short hop; it binds on about 1% of region DayHikes (a village loop with a fountain every few hundred metres), so it hardly matters. |

The Water FacetScore (`water_score`) is the plain mean of the two. PRD
starting constants stay; the data does not argue for other values.

**What the data holds.** Springs count since the WaterSource re-ingest, and they
matter most in the mountains: `05` prints the split by highest point (57% of the
hikes above 1,500 m pass no WaterSource, against 73% when only drinking water
counted). Streams, troughs and unmapped springs are still missing, so the Facet
compares what OSM maps, not what is in the ground. A DryStretch is also the
hike's whole length for the 60% that pass no WaterSource, so for those the
Water Facet partly repeats Effort's length; the count is the part that is new.

Result on the real region (below): the main similar pair scores 0.97 and the
dissimilar one 0.05. The four extra pairs separate with a margin of 0.71: the
lowest similar score (two dry karst walks, 0.92) against the highest
dissimilar (0.21, the hike whose only WaterSource is at its very end against a
well-watered town walk). Both soft targets are met.

In [ ]:
WATER_CAP = 3
WATER_COUNT_D = 3
DRY_STRETCH_FLOOR_M = 1_000


def capped_water_count(count):
    return None if pd.isna(count) else min(count, WATER_CAP)


def water_score(a, b):
    return facet_score(
        [
            tolerance(capped_water_count(a["water_count"]), capped_water_count(b["water_count"]), d=WATER_COUNT_D),
            ratio(a["dry_stretch_m"], b["dry_stretch_m"], floor=DRY_STRETCH_FLOOR_M),
        ]
    )


measure_spread(
    "water_count",
    lambda a, b: tolerance(capped_water_count(a), capped_water_count(b), d=WATER_COUNT_D),
    f"tolerance cap={WATER_CAP} D={WATER_COUNT_D}",
)
measure_spread("dry_stretch_m", lambda a, b: ratio(a, b, floor=DRY_STRETCH_FLOOR_M), f"ratio floor={DRY_STRETCH_FLOOR_M}")
print(f"water_count_share\t{facts['water_count'].clip(upper=WATER_CAP).value_counts(normalize=True).sort_index().round(3).to_dict()}")
print(f"dry_stretch_under_floor_share\t{(facts['dry_stretch_m'] < DRY_STRETCH_FLOOR_M).mean():.3f}")

### Water verification (real region)

Expected before computing:

- **Similar:** "Stara Fužina - Dom v Vojah" (Bohinj, 4 WaterSources) and
  "Kanjon Velika Paklenica" (Velebit, 7): well-watered 3.4 and 6.5 km walks.
  Both have the count at the cap (three or more) and a DryStretch of about
  1.8–1.9 km, so they should score about 0.95 or more. They are in different
  mountains, which Water ignores.
- **Dissimilar:** "Kanjon Velika Paklenica" against "Strogi rezervat prirode
  Bijele i Samarske stijene", a 19 km karst walk in Gorski kotar with no
  WaterSource at all: count 3 against 0 scores 0, and 1.8 km of dry stretch
  against 19.3 km about 0.09, so the Facet should be near 0.05.

The dry partner was "Put preko vrha Dinare" until springs were counted; it now
passes two springs on its 18 km ridge (DryStretch 10.9 km), so it is no longer a
hike without water.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("water pairs skipped on the fixture")
else:
    water_pairs = pair_rows(
        ["Stara Fužina - Dom v Vojah", "Kanjon Velika Paklenica"],
        ["Kanjon Velika Paklenica", "Strogi rezervat prirode Bijele i Samarske stijene"],
    )
    show_pairs(water_pairs, ["length_m", "water_count", "dry_stretch_m"], "Water pairs")
    verify_pairs("water", water_score, water_pairs)

### More Water pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** two dry karst walks, "Strogi rezervat prirode Bijele i Samarske
  stijene" (Gorski kotar, 19 km) and "Kurtagić - Vaganj - Vrdovo" (the Vrdovo
  karst plateau on Kamešnica, 16 km): no WaterSource at all, so the count agrees
  and the DryStretch is the whole walk, 19.3 against 16.1 km (ratio 0.83, Facet
  about 0.92). And two short town walks, "Čez Ljubljanski grad" (7
  points) and "Sentiero Tabor / Pešpot Tabor" (4): both are at the cap and
  below the 1 km floor, so they should score 1.
- **Dissimilar:** "Sentiero natura / Pešpot za Miramar", a Trieste seafront
  walk with 19 points, against "Wanderweg 338", a 19.5 km Styrian ridge walk
  with none (near 0). And "Lendweg" (7 points, 5 km walk, DryStretch 1.4 km)
  against "Mala Kočevska planinska pot", a 17 km walk near Kočevje with one
  WaterSource at its very end, which cuts nothing: its DryStretch is its whole
  length. Count alone gives 0.33; the DryStretch (1.4 against 17.4 km, about
  0.08) pulls the Facet down to about 0.2.

Two pairs were re-picked once springs were counted: "Put preko vrha Dinare"
now passes two springs, and "Z (Nemesvita – Büdös-kút – Vonyarcvashegy)" three
WaterSources (Büdös-kút is a spring), so neither plays its old part.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one. Skipped on the fixture.

In [ ]:
WATER_MORE = [
    ("similar", "Strogi rezervat prirode Bijele i Samarske stijene", "Kurtagić - Vaganj - Vrdovo"),
    ("similar", "Čez Ljubljanski grad", "Sentiero Tabor / Pešpot Tabor"),
    ("dissimilar", "Sentiero natura / Pešpot za Miramar", "Wanderweg 338"),
    ("dissimilar", "Lendweg", "Mala Kočevska planinska pot"),
]

if on_fixture:
    print("more water pairs skipped on the fixture")
else:
    more_rows = pair_rows(sorted({n for _, a, b in WATER_MORE for n in (a, b)}), [])
    more_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "counts": f"{int(more_rows.at[a, 'water_count'])} / {int(more_rows.at[b, 'water_count'])}",
            "dry_km": f"{more_rows.at[a, 'dry_stretch_m'] / 1000:.1f} / {more_rows.at[b, 'dry_stretch_m'] / 1000:.1f}",
            "score": water_score(more_rows.loc[a], more_rows.loc[b]),
        }
        for expected, a, b in WATER_MORE
    )
    print(more_table.round(3).to_string())
    by_group = more_table.groupby("expected")["score"]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    verify("water", "more_pairs_margin", margin, margin > 0)

### Water synthetic checks

Same three checks. For reversed, the Water facts are rebuilt from the reversed
vertex chain: the Corridor is measured to the line and each point is placed by
its closest place on it, so every position becomes length minus position and
the DryStretch stays the same.

In [ ]:
synthetic_checks(
    "water",
    water_score,
    water_facts(dayhikes, vertices, amenities),
    water_facts(dayhikes, backward_vertices, amenities),
)

## Facilities

One measure: presence over the five Facilities flags. The facts are built as in
`05`: a Hut, a restaurant or cafe and a Shelter within 100 m of the line, and a
restaurant or cafe and parking within 500 m of the Catalog Trailhead. The
Corridor is measured to the line, the Trailhead distance is a plain distance,
and a Shelter never counts as a Hut.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| five flags (`hut_along`, `restaurant_cafe_along`, `shelter_along`, `restaurant_cafe_trailhead`, `parking_trailhead`) | presence | none | The share of the five flags on which the two DayHikes agree. Both-absent is agreement, so two hikes that both lack a Hut agree on that point. The flags are common enough to carry information (a Hut along 22% of region DayHikes, a restaurant or cafe along 32%, a Shelter 16%, a restaurant or cafe at the Trailhead 35%, parking 56%) and rare enough that most flags are false on both sides. |

The Facilities FacetScore (`facilities_score`) is that one number, so it comes
in fifths: 0, 0.2, 0.4, 0.6, 0.8 or 1. The flags are equally weighted; the data
gives no reason to favour one, and the Hiker's FacetWeights act on the Facet as
a whole.

**Presence is lenient.** Agreement on absence counts, so a Hut hike with no
other facility and a hike with nothing at all still agree on four flags out of
five and score 0.8. A random pair of region DayHikes scores a median 0.6. Read
the Facet by its extremes (all five alike against all five different) and not
by the dissimilar soft target alone, which a single missing Hut can miss
without the Facet being wrong. The same holds for the presence measures in
Landscape.

Result on the real region (below): the main similar pair scores 1.00 and the
dissimilar one 0.00; both soft targets are met. The four extra pairs separate
with a margin of 0.20: the lowest similar score is 1.00 (two hut-only hikes, two
lunch-stop walks) and the highest dissimilar is 0.80, the hut hike against the
no-facility hike, which differ on the Hut flag alone. The hut hike against the
lunch walk scores 0.20. The 0.4 soft target is therefore missed for the
one-flag case by design, not by error. A random pair's quartiles are 0.4, 0.6
and 0.8.

In [ ]:
FLAGS = list(FACILITIES_FLAG_COLUMNS)


def facilities_score(a, b):
    return facet_score([presence([a[c] for c in FLAGS], [b[c] for c in FLAGS])])


for flag in FLAGS:
    agree = np.mean([facts.at[a, flag] == facts.at[b, flag] for a, b in zip(left_ids, right_ids)])
    print(
        f"measure\t{flag}\tpresence\ttrue_share={facts[flag].mean():.3f}\t"
        f"missing={facts[flag].isna().mean():.3f}\trandom_pair_agree={agree:.2f}"
    )
pair_scores = [facilities_score(facts.loc[a], facts.loc[b]) for a, b in zip(left_ids, right_ids)]
q = np.quantile(pair_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist()
print(f"measure\tfacilities_score\tpresence\trandom_pair_q05,q25,q50,q75,q95={q}")
print("flags_set_share\t" + str(facts[FLAGS].sum(axis=1).value_counts(normalize=True).sort_index().round(3).to_dict()))

### Facilities verification (real region)

Expected before computing:

- **Similar:** "Cojzova koča - Skuta" and "Krekova koča - Altemaver - Kremant".
  Two Kamnik Alps hut hikes (to 2,490 m and 1,670 m): a Hut and a Shelter along
  the way, no restaurant or cafe, nothing at the Trailhead. The flags are
  identical, so the Facet should be 1. They are different routes of different
  length, which Facilities ignores.
- **Dissimilar:** "Šmartno - Koča pod Kremžarjevim vrhom" and "Draževnik -
  Gradišče". The first has all five flags (a Hut, a restaurant or cafe and a
  Shelter along the way, a restaurant or cafe and parking at the Trailhead); the
  second has none. They agree on nothing, so the Facet should be 0.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("facilities pairs skipped on the fixture")
else:
    facilities_pairs = pair_rows(
        ["Cojzova koča - Skuta", "Krekova koča - Altemaver - Kremant"],
        ["Šmartno - Koča pod Kremžarjevim vrhom", "Draževnik - Gradišče"],
    )
    show_pairs(facilities_pairs, ["length_m", *FLAGS], "Facilities pairs")
    verify_pairs("facilities", facilities_score, facilities_pairs)

### More Facilities pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** two hut-only hikes, "Okrešelj - Kamniško sedlo" and "Hudičev
  boršt - Kališče": a Hut along the way, nothing else, identical flags (1). And
  two walks with a lunch stop, "Kulinarische Weinberg-Wanderung" and "Wanderweg
  343": a restaurant or cafe along the way and a restaurant or cafe and parking
  at the Trailhead, no Hut and no Shelter. Identical flags (1).
- **Dissimilar:** the hut hike "Okrešelj - Kamniško sedlo" against the
  lunch walk "Kulinarische Weinberg-Wanderung" (a Hut and no meal against a meal
  and no Hut, four flags differ; about 0.2). And the hut hike against the
  no-facility hike "Mali Alan - Dušice" (a Hut and nothing else against nothing
  at all; only the Hut flag differs, so about 0.8). This second pair is the
  weak case named above: the Facet separates it from the similar pairs, but not
  by the 0.4 soft target.

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one. Skipped on the fixture.

In [ ]:
FACILITIES_MORE = [
    ("similar", "Okrešelj - Kamniško sedlo", "Hudičev boršt - Kališče"),
    ("similar", "Kulinarische Weinberg-Wanderung", "Wanderweg 343"),
    ("dissimilar", "Okrešelj - Kamniško sedlo", "Kulinarische Weinberg-Wanderung"),
    ("dissimilar", "Okrešelj - Kamniško sedlo", "Mali Alan - Dušice"),
]

if on_fixture:
    print("more facilities pairs skipped on the fixture")
else:
    more_rows = pair_rows(sorted({n for _, a, b in FACILITIES_MORE for n in (a, b)}), [])
    more_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "flags_a": "".join("1" if more_rows.at[a, c] else "0" for c in FLAGS),
            "flags_b": "".join("1" if more_rows.at[b, c] else "0" for c in FLAGS),
            "score": facilities_score(more_rows.loc[a], more_rows.loc[b]),
        }
        for expected, a, b in FACILITIES_MORE
    )
    print(more_table.round(3).to_string())
    by_group = more_table.groupby("expected")["score"]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    verify("facilities", "more_pairs_margin", margin, margin > 0)

### Facilities synthetic checks

Same three checks. For reversed, the Facilities facts are rebuilt from the
reversed vertex chain: the Corridor is measured to the line and the Trailhead
distance does not use the line at all, so the walking direction must not matter.

In [ ]:
synthetic_checks(
    "facilities",
    facilities_score,
    facilities_facts(dayhikes, vertices, amenities),
    facilities_facts(dayhikes, backward_vertices, amenities),
)

## Trail

Two measures: the SurfaceMix and the TarmacApproach. The facts are built as in
`05` (four length fractions over the classified in-chain member ways, from the
`surface` tag else the highway kind; the paved length from the chain end
nearest the Trailhead until the first non-paved way).

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| SurfaceMix (four shares) | mix | none | Scores the shared share: all natural path against half natural, half gravel is 0.5. A random pair of region DayHikes shares a median 0.49 (natural is half of the average hike, gravel 28%, paved 22%, rock 1%). |
| `tarmac_approach_m` | tolerance | D = 2 km | 58% of region DayHikes have no approach at all (the first way is unpaved); the 75th percentile is 0.42 km, the 90th 1.3 km, the 95th 2.3 km and the maximum 14.8 km. D = 2 km scores a difference of 2 km or more as 0, and 0.5 km of road against none as 0.75. A random pair's median is 0.88 because most pairs both have none. |

The Trail FacetScore (`trail_score`) is the plain mean of the two. The PRD
starting constant (D = 2 km) stays; the data does not argue for another (a
median random pair would score 0.75 at D = 1 km and 0.95 at D = 5 km, and the
95th percentile is 2.3 km).

Scope of the mix: 47% of walked ways carry a listed `surface` tag; the other 53%
(tracks and paths mostly, then residential and unclassified roads) come from
the highway kind, which is why the mix stays defined for nearly every hike.
Mixes are over the classified length, so a hike's unclassified bits (24 ways
in the region) do not count against it.

A loop has both chain ends at the Trailhead, so either direction is a fair
start: its TarmacApproach is the shorter of the two.

Result on the real region (below): the main similar pair scores 1.00 and the
dissimilar one 0.00 (a forest path against a promenade share nothing, and
differ by 3.6 km of road). The four extra pairs separate with a margin of 0.70:
the lowest similar score (the two road-approach hikes, 0.71) against the
highest dissimilar (0.00).

In [ ]:
TARMAC_D_M = 2_000
SURFACE = list(SURFACE_SHARE_COLUMNS)


def trail_score(a, b):
    return facet_score(
        [
            mix([a[c] for c in SURFACE], [b[c] for c in SURFACE]),
            tolerance(a["tarmac_approach_m"], b["tarmac_approach_m"], d=TARMAC_D_M),
        ]
    )


mix_scores = [
    mix([facts.at[a, c] for c in SURFACE], [facts.at[b, c] for c in SURFACE])
    for a, b in zip(left_ids, right_ids)
]
mix_scores = [s for s in mix_scores if s is not None]
q = np.quantile(mix_scores, [0.05, 0.25, 0.5, 0.75, 0.95]).round(2).tolist() if mix_scores else []
print(
    f"measure\tsurface_mix\tmix\tmissing={facts[SURFACE].isna().any(axis=1).mean():.3f}\t"
    f"random_pair_q05,q25,q50,q75,q95={q}"
)
for column in SURFACE:
    print(f"share\t{column}\tmean={facts[column].mean():.3f}\tq95={facts[column].quantile(0.95):.3f}")
print(f"tarmac_zero_share\t{(facts['tarmac_approach_m'] == 0).mean():.3f}")
measure_spread("tarmac_approach_m", lambda a, b: tolerance(a, b, d=TARMAC_D_M), f"tolerance D={TARMAC_D_M}")

### Trail verification (real region)

Expected before computing:

- **Similar:** "Dom Paklenica - Jurline" and "Stap - Jelova Ruja". Both are
  all-natural forest paths in the Velebit (0 m of road before the trail), so
  the mix is the same and the approach is the same.
- **Dissimilar:** "Krč - Lišćani bunar" and "314 Cres – Lungomare". The first
  is a forest path with no road; the second is a seaside promenade, paved from
  end to end, with 3.6 km of tarmac approach.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("trail pairs skipped on the fixture")
else:
    trail_pairs = pair_rows(
        ["Dom Paklenica - Jurline", "Stap - Jelova Ruja"],
        ["Krč - Lišćani bunar", "314 Cres – Lungomare"],
    )
    show_pairs(trail_pairs, [*SURFACE, "tarmac_approach_m"], "Trail pairs")
    verify_pairs("trail", trail_score, trail_pairs)

### More Trail pairs (real region)

Expected before computing. Two that should be similar, two that should not:

- **Similar:** "Planinarska staza Medvednica 18" and "Planinarska staza
  Medvednica 36": two all-gravel tracks on the same mountain, no road
  approach. "Jelovec - Žavcarjev vrh" and "Bresternica - Žavcarjev vrh":
  both leave a Pohorje village by road (3.6 and 4.5 km of tarmac) and finish
  on forest tracks, half paved.
- **Dissimilar:** the forest path "Dom Paklenica - Jurline" against the
  asphalt "Welschlauf Südsteiermark" (13.6 km of road, all paved), and the
  gravel "Planinarska staza Medvednica 18" against the paved "Stepinčev put 2.
  etapa" (3.5 km of road approach).

The check is the same as the main pair: the lowest similar score must be above
the highest dissimilar one. Skipped on the fixture.

In [ ]:
TRAIL_MORE = [
    ("similar", "Planinarska staza Medvednica 18", "Planinarska staza Medvednica 36"),
    ("similar", "Jelovec - Žavcarjev vrh", "Bresternica - Žavcarjev vrh"),
    ("dissimilar", "Dom Paklenica - Jurline", "Welschlauf Südsteiermark"),
    ("dissimilar", "Planinarska staza Medvednica 18", "Stepinčev put 2. etapa"),
]

if on_fixture:
    print("more trail pairs skipped on the fixture")
else:
    more_rows = pair_rows(sorted({n for _, a, b in TRAIL_MORE for n in (a, b)}), [])
    more_table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "surface_mix": mix([more_rows.at[a, c] for c in SURFACE], [more_rows.at[b, c] for c in SURFACE]),
            "tarmac": tolerance(more_rows.at[a, "tarmac_approach_m"], more_rows.at[b, "tarmac_approach_m"], d=TARMAC_D_M),
            "score": trail_score(more_rows.loc[a], more_rows.loc[b]),
        }
        for expected, a, b in TRAIL_MORE
    )
    print(more_table.round(3).to_string())
    by_group = more_table.groupby("expected")["score"]
    margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
    verify("trail", "more_pairs_margin", margin, margin > 0)

### Trail synthetic checks

Same three checks. For reversed, the Trail facts are rebuilt from the reversed
vertex chain: the ways are put back in walking order and the approach starts at
the end nearest the Trailhead, so the walking direction must not matter.

In [ ]:
synthetic_checks(
    "trail",
    trail_score,
    trail_facts(dayhikes, vertices, ways),
    trail_facts(dayhikes, backward_vertices, ways),
)

## Combined score: one Seed, two FacetWeights sets

Every region DayHike gets all eight FacetScores against one Seed, computed
once. FacetWeights only change how they combine (`combined_score`: Σ wᶠ·sᶠ / Σ wᶠ
over the Facets that are scored and have wᶠ > 0, plus coverage). The Seed is
"Aljažev dom - Kredarica - Triglav" (on the fixture, "Ridge Walk"): a 4.6 km,
1,850 m climb from 1,000 m to the Kredarica hut at 2,500 m, with a Hut along
the way and parking at the Trailhead, and no restaurant, cafe or Shelter.

| Facet | Set A `facilities_heavy` | Set B `altitude_heavy` |
| --- | --- | --- |
| Effort | 0 | 20 |
| Altitude | 30 | 60 |
| Profile | 0 | 0 |
| Landscape | 20 | 20 |
| Weather | 0 | 0 |
| Water | 0 | 0 |
| Facilities | 50 | 0 |
| Trail | 0 | 0 |

Set A is the PRD example: "I care about huts and how high it goes, not about
the trees." Set B makes Altitude the main theme. It keeps Landscape at 20 and
gives Facilities' weight to Effort, so a hike that climbs as high from as high,
with the same work, wins whatever its Huts.

The tables show all eight FacetScores, including the ones a set weights 0, so
you can see what each set ignores. `weather_score` uses the `QUERY_MONTH` that
was set when the Weather cell ran. After you change it, re-run that cell and
then this one.

In [ ]:
FACET_SCORES = {
    "effort": effort_score,
    "altitude": altitude_score,
    "profile": profile_score,
    "landscape": landscape_score,
    "weather": weather_score,  # binds QUERY_MONTH when the Weather cell ran
    "water": water_score,
    "facilities": facilities_score,
    "trail": trail_score,
}
FACET_WEIGHTS = {
    "facilities_heavy": {
        "effort": 0, "altitude": 30, "profile": 0, "landscape": 20,
        "weather": 0, "water": 0, "facilities": 50, "trail": 0,
    },
    "altitude_heavy": {
        "effort": 20, "altitude": 60, "profile": 0, "landscape": 20,
        "weather": 0, "water": 0, "facilities": 0, "trail": 0,
    },
}
SEED = "Ridge Walk" if on_fixture else "Aljažev dom - Kredarica - Triglav"


def seed_row(name):
    rows = facts[facts["name"] == name]
    assert len(rows) == 1, f"Seed name must be unique: {name}"
    return rows.iloc[0]


def facet_scores_against(seed, candidates):
    """All eight FacetScores of each candidate against one Seed (NaN = missing)."""
    scores = {
        dayhike_id: {facet: fn(seed, row) for facet, fn in FACET_SCORES.items()}
        for dayhike_id, row in candidates.iterrows()
    }
    return pd.DataFrame.from_dict(scores, orient="index", columns=list(FACET_SCORES)).astype(float)


def rank(scores, weights):
    """Combined score and coverage per candidate, best first."""
    combined = [combined_score(row.to_dict(), weights) for _, row in scores.iterrows()]
    ranking = scores.assign(
        score=[np.nan if s is None else s for s, _ in combined],
        coverage=[c for _, c in combined],
    )
    ranking.insert(0, "name", facts.loc[scores.index, "name"])
    return ranking.sort_values("score", ascending=False)


seed = seed_row(SEED)
seed_scores = facet_scores_against(seed, facts.drop(index=seed.name))
print(f"seed\t{SEED}\tquery_month\t{QUERY_MONTH}")
print(
    facts.loc[[seed.name], ["length_m", "elevation_gain_m", "highest_m", "lowest_m", *FLAGS, "tree_share", "bare_share"]]
    .round(2)
    .T.to_string()
)
rankings = {}
for set_name, weights in FACET_WEIGHTS.items():
    rankings[set_name] = rank(seed_scores, weights)
    top = rankings[set_name].head(5)
    print(f"\n{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0))
    print(top.round(3).to_string())
    for _, row in top.iterrows():
        print(f"rank\t{set_name}\t{row['name']}\t{row['score']:.3f}\t{row['coverage']:.2f}")
shared = set(rankings["facilities_heavy"].head(5).index) & set(rankings["altitude_heavy"].head(5).index)
print(f"top5_shared\t{len(shared)}")

In [ ]:
def show_ranking_map(seed, ranking, title):
    """The Seed (thick orange) and the top five (blue, numbered by rank) on one map."""
    top = gpd.GeoDataFrame(facts.loc[ranking.head(5).index], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
    seed_line = gpd.GeoDataFrame([seed], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
    fig, ax = plt.subplots(figsize=(7, 7))
    top.plot(ax=ax, color="#1d4ed8", linewidth=2)
    seed_line.plot(ax=ax, color="#c2410c", linewidth=4)
    for number, (_, row) in enumerate(top.iterrows(), start=1):
        point = row.geometry.representative_point()
        ax.annotate(f"{number}. {row['name']}", (point.x, point.y), fontsize=7, color="#1d4ed8")
    point = seed_line.geometry.iloc[0].representative_point()
    ax.annotate(f"Seed: {seed['name']}", (point.x, point.y), fontsize=8, color="#c2410c", weight="bold")
    try:
        import contextily as cx

        cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
    except Exception as error:  # offline: the lines alone are still the map
        print("basemap skipped", type(error).__name__)
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()
    fig.tight_layout()
    show_fig(fig)


for set_name, weights in FACET_WEIGHTS.items():
    show_ranking_map(
        seed,
        rankings[set_name],
        f"{set_name}: " + ", ".join(f"{facet} {w}" for facet, w in weights.items() if w > 0),
    )

### Why the two top fives differ

Result on the real region (above): the two top fives share two DayHikes,
"Krma - Staničev dom" (1st under A, 3rd under B) and "Hanzova pot" (4th, 2nd).
The other places follow the weights:

- **Set A puts half its weight on Facilities.** The Seed has a Hut along the
  way and parking at the Trailhead, and nothing else. Only 5.5% of region
  DayHikes have exactly those five flags (Facilities 1.0), and all of set A's
  top five are among them. Effort weighs 0 in A, so "101" ranks fifth: it is
  12.4 km long against the Seed's 4.6 km (Effort 0.39).
- **Set B puts 60 on Altitude and 20 on Effort.** "Tamar - Jalovška škrbina -
  Jalovec" is first under B. It goes from 1,110 to 2,580 m (Altitude 0.91)
  and climbs 1,810 m over 5.5 km (Effort 0.72). It has no parking at the
  Trailhead, so its Facilities score is 0.8. Under A that costs it 0.1 and it
  ranks 17th. "Gozdec - Dom Petra Skalarja - Prestreljenik" (Altitude 0.94) has
  a cafe and a Shelter the Seed lacks, so Facilities is 0.6; it is 5th under B
  and 241st under A.
- **It works the other way too.** "relation:4074408" is 2nd under A, with
  matching flags, but it tops out at 2,030 m against the Seed's 2,510 m
  (Altitude 0.74). Under B it drops to 33rd.

Coverage is 1.00 for every candidate in both sets: no Facet is missing in this
region (see Coverage below).

## Several Seeds: the maximum rule

With several Seeds, a candidate's score is its best combined score against any
one Seed (`multi_seed_score`, as ADR-0003 does for production Ranking), never
the mean. Two dissimilar Seeds are used here: the alpine "Aljažev dom -
Kredarica - Triglav" and the Hvar coast climb "Zavala - Sv. Ante" (on the
fixture, "Ridge Walk" and "Valley Path"), both with set B weights. Every
candidate is scored against each Seed. Each `multi_seed` line prints the score
followed by the two per-Seed scores, so you can check by eye that the score is
the larger one; the cell also asserts this for every candidate. The `mean`
column shows what averaging would give instead.

Result on the real region (below): the coastal Seed wins 2,879 candidates and
the alpine Seed 1,527. The top ten are all coastal or low-hill walks
("Francuska cesta": 0.909 against Zavala, 0.187 against Kredarica), because
the region has many more DayHikes like Zavala than like Kredarica. The first
candidate the alpine Seed wins ranks 36th: "Tamar - Jalovška škrbina - Jalovec"
at 0.824, the same score it gets against Kredarica alone. Each score belongs
to one Seed and is never stitched from both. Averaging would rank hikes by how
little they resemble either Seed: every top mean is about 0.55, and "Senj -
Planinarska kuća Sijaset" (0.29 / 0.81) enters the top five.

In [ ]:
SEEDS = ["Ridge Walk", "Valley Path"] if on_fixture else ["Aljažev dom - Kredarica - Triglav", "Zavala - Sv. Ante"]
MULTI_WEIGHTS = FACET_WEIGHTS["altitude_heavy"]

seeds = [seed_row(name) for name in SEEDS]
candidates = facts.drop(index=[s.name for s in seeds])
multi = pd.DataFrame(
    {"name": candidates["name"], **{s["name"]: rank(facet_scores_against(s, candidates), MULTI_WEIGHTS)["score"] for s in seeds}}
)
multi["score"] = pd.Series(
    [multi_seed_score([a, b]) for a, b in zip(multi[SEEDS[0]], multi[SEEDS[1]])], index=multi.index, dtype=float
)
multi["mean"] = multi[SEEDS].mean(axis=1)
multi = multi.sort_values("score", ascending=False)
assert np.allclose(multi["score"], multi[SEEDS].max(axis=1), equal_nan=True), "score must be the higher of the two"

print("seeds", SEEDS, "weights", {facet: w for facet, w in MULTI_WEIGHTS.items() if w > 0})
print(multi.head(10).round(3).to_string())
for _, row in multi.head(10).iterrows():
    print(f"multi_seed\t{row['name']}\t{row['score']:.3f}\t{row[SEEDS[0]]:.3f}\t{row[SEEDS[1]]:.3f}")
won_by_first = (multi[SEEDS[0]] >= multi[SEEDS[1]]).to_numpy()
for name, won in ((SEEDS[0], won_by_first), (SEEDS[1], ~won_by_first)):
    position = int(np.argmax(won)) + 1 if won.any() else None
    print(f"multi_seed_won\t{name}\tcandidates={int(won.sum())}\tbest_rank={position}")
by_mean = multi.sort_values("mean", ascending=False).head(5)
print("top 5 if the two scores were averaged instead:")
print(by_mean.round(3).to_string())

## Coverage: how often each Facet is missing

A Facet is missing for a DayHike when the DayHike cannot be scored on it even
against itself (every measure missing). The Facet then drops out of the
combined score, and coverage falls below 1. The cell prints that share for each
Facet (`facet_missing`), then every fact that is ever missing, then the share of
ranked candidates whose coverage is below 1 under each FacetWeights set.

Result on the real region (below): no Facet is missing for any of the 4,408
DayHikes, so coverage is 1.00 in both rankings. Two facts are missing inside a
Facet and drop out of its mean:

- Difficulty, on 45.6% of DayHikes. Effort then rests on length and gain.
- Southness, on 1.1% (49 flat lowland walks with no point steeper than 5°).
  Weather then rests on the three climate values.

Climate is never missing, for the regime or for any month. The Facilities flags
and the Landscape presence facts are never missing by construction (no
amenity in range means False). On the fixture, southness is always missing (no
DEM tile) and Weather still scores. Coverage will matter outside the sampled
rasters, in the whole-of-Europe phase, but not in this region.

In [ ]:
MEASURE_COLUMNS = {
    "effort": ["length_m", "elevation_gain_m", "difficulty_rank"],
    "altitude": ["highest_m", "lowest_m"],
    "profile": ["profile_shape"],
    "landscape": [*LANDCOVER, "peak_summited", "viewpoint_along", "peaks_around"],
    "weather": [*climate_slice(QUERY_MONTH), "southness"],
    "water": ["water_count", "dry_stretch_m"],
    "facilities": FLAGS,
    "trail": [*SURFACE, "tarmac_approach_m"],
}

# A Facet is missing for a DayHike when it cannot be scored even against itself.
unscored = {facet: 0 for facet in FACET_SCORES}
for _, row in facts.iterrows():
    for facet, fn in FACET_SCORES.items():
        unscored[facet] += fn(row, row) is None
for facet, count in unscored.items():
    print(f"facet_missing\t{facet}\t{count / len(facts):.3f}")

measure_missing = pd.DataFrame(
    [
        {"facet": facet, "fact": column, "missing": facts[column].isna().mean()}
        for facet, columns in MEASURE_COLUMNS.items()
        for column in columns
    ]
)
print(measure_missing[measure_missing["missing"] > 0].round(3).to_string(index=False))
print(f"facts_never_missing\t{(measure_missing['missing'] == 0).sum()} of {len(measure_missing)}")
for set_name, ranking in rankings.items():
    print(f"coverage_below_1\t{set_name}\t{(ranking['coverage'] < 1).mean():.3f}")

## Verification summary

This table collects every `verify` line above (Facet, check, score,
PASS/FAIL). `verify` records each line as it prints it. Below the table: the
count of passes and failures, each failure, and each missed soft target (from
the `soft` lines). On the fixture only the identity, symmetry and reversed
checks run (24 lines).

In [ ]:
summary = pd.DataFrame(
    [{"facet": facet, "check": check, "score": score, "result": result} for (facet, check), (score, result) in VERIFY.items()]
)
print(summary.round(3).to_string(index=False))
passes = int((summary["result"] == "PASS").sum())
print(f"verify_summary\tpass={passes}\tfail={len(summary) - passes}")
for row in summary[summary["result"] == "FAIL"].itertuples():
    print(f"verify_failed\t{row.facet}\t{row.check}\t{row.score:.3f}")
for (facet, check), (score, target, met) in SOFT.items():
    if not met:
        print(f"soft_missed\t{facet}\t{check}\t{score:.3f}\t{target}")

### Reading the summary (real region)

45 checks: 45 PASS, 0 FAIL. They are 16 named-pair lines (a similar and a
dissimilar pair for each of the eight Facets), 5 more-pairs margins (Landscape,
Weather, Water, Facilities, Trail) and 24 synthetic checks (identity, symmetry,
reversed for each Facet).

**Failures:** none.

**Missed soft targets.** These are known design points, not failures. Each
pair still scores below its similar pair:

- **Profile, dissimilar pair 0.684** (target ≤ 0.4). Two ProfileShapes that both
  go up and down share much of the unit square: a random pair's median is
  0.77, and even flat against a straight climb scores 0.52. The verdict rests on
  similar > dissimilar (0.984 against 0.684).
- **Landscape, forest against open karst 0.58** ("Predmeja - Iztokova koča"
  against "Zahum Idovac", in the more-pairs table). The LandCover mixes share
  only 0.04, but neither hike has a summit or a Viewpoint. Both-absent flags
  agree, so 2 of the 4 measures score 1.
- **Facilities, one-flag difference 0.8** ("Okrešelj - Kamniško sedlo" against
  "Mali Alan - Dušice", in the more-pairs table). Presence is lenient: a Hut
  hike with no other facility and a hike with none agree on 4 of 5 flags. The
  margin is still 0.2 (PASS).

## For review

**Constants changed from the PRD starting constants: none.** Every constant in
the PRD table was kept after looking at the region's spread (each Facet section
gives the reason). Where the PRD left a value open, this is how it was set:

| Constant | Value | Reason |
| --- | --- | --- |
| Weather: solar radiation D | 5,000 kJ m⁻² day⁻¹ | The PRD says "from the region's spread". This is the 5th–95th percentile spread of the June–August regime (17,779–22,684). The same D is kept for single months, so winter solar scores sit higher (January's spread is 2,015). |
| SolarAspect: sampled points | about 32 vertices per DayHike, counted from the nearer chain end | Cheap on the region, and the same set of points in both walking directions, so the reversed check is exact. |
| SolarAspect: east and north offsets | 2 FABDEM cells (2 arc-seconds, about 44 m east and 62 m north) | The PRD says "small offsets". A 30 m offset often reads the same 31 m cell; two cells always read a different cell at a known distance. |
| SolarAspect: slope floor | 5° | The PRD says "a few degrees". One or two metres of FABDEM noise over 44–62 m is 1–3°. |
| Corridor: densify step | radius / 10 (10 m for the 100 m Corridor) | Distance is measured to the line, and the error is under 0.2% of the radius. |
| Profile: DTW band | round(0.1 × 64) = 6 points | The PRD's 10% band, rounded to whole points. |

**Profile method kept: lock-step.** The rule was to keep DTW only if it
separates the verification pairs better, and it does not. The margin (lowest
similar minus highest dissimilar) is 0.148 for lock-step against 0.121 for DTW,
and the mean gap is 0.303 against 0.258. DTW lifts the dissimilar pairs more
than the similar ones, and it moves the random-pair median from 0.77 to 0.85.

**Rules chosen in this phase**

- Elevation gain is direction-free: max(climb, descent) along the vertex chain,
  instead of the Catalog's `elevation_gain_m` in stored direction. OSM way order
  is arbitrary, so the stored figure fails the reversed check (fixture Ridge
  Walk: 450 m forward, 250 m reversed).
- The water count is capped at 3 in the 05a measure. The `05` fact is the
  true count.
- TarmacApproach on a loop is the shorter of the two ends. `_link` highways
  count as roads. An unlisted `surface` falls back to the highway kind.
- `presence` drops a flag that is missing on either side.

**Open design questions**

- **FABDEM vertex gain may be inflated by noise.** "Vršič - Razor" shows
  1,997 m of gain on a 902 m range. Smoothing the ElevationProfile before gain
  (and before ProfileShape) is not done yet.
- **Southness on mostly flat walks rests on a few sloped points** (embankments,
  road cuts). A minimum count of sloped points is an option if this turns out
  noisy in Ranking.
- **Only mapped water counts.** Springs, taps and water points are
  WaterSources since the re-ingest; streams, troughs and unmapped springs are
  not, so Water can still understate the water on backcountry hikes.
- **Which gain does the Catalog keep?** The direction-free gain (max of climb
  and descent) differs from the Catalog's stored-direction value on
  point-to-point hikes stored downhill. The promoted Catalog column has to pick
  one.
- **Lock-step over DTW.** Kept on the six verification pairs. Revisit if
  hikes whose top sits at a different point along the walk should match.
- **Presence is lenient** (both-absent agrees) in Landscape and Facilities.
  The ranking demo still separates (only 5.5% of candidates match the Seed's
  five flags), so this is revisited only if FacetWeights need sharper
  separation.
- **DryStretch is the whole length for the 60% of hikes with no WaterSource**, so
  for them Water partly repeats Effort's length.